In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/erdemyavuz55/drugscom/drugsComTest_raw.csv
/kaggle/input/datasets/erdemyavuz55/drugscom/drugsComTrain_raw.csv


In [2]:
!pip install numpy==1.21.6 packaging==22 shapely==2.0.1 scikit-learn==1.3.1 matplotlib==3.8.0 scipy==1.14.0
!pip install gensim nltk

ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement numpy==1.21.6 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17.0, 1.17.1, 1.17.2, 1.17.3, 1.17.4, 1.17.5, 1.18.0, 1.18.1, 1.18.2, 1.18.3, 1.18.4, 1.18.5, 1.19.0, 1.19.1, 1.19.2, 1.19.3, 1.19.4, 1.19.5, 1.20.0, 1.20.1, 1.20.2, 1.20.3, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.22.2, 

In [3]:
# Import Needed libaries and packages
import numpy as np
import pandas as pd
import html
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [4]:
# Importing data
train_data = pd.read_csv('/kaggle/input/datasets/erdemyavuz55/drugscom/drugsComTrain_raw.csv')
test_data = pd.read_csv('/kaggle/input/datasets/erdemyavuz55/drugscom/drugsComTest_raw.csv')

In [5]:
# Print the total number os nan in the review column
print(train_data['review'].isna().sum())
print(test_data['review'].isna().sum())

0
0


In [6]:
# Print the first 5 instances from the data
print(train_data.head()['review'])
print("-------------------------------------------------------------")
print(test_data.head()['review'])

0    "It has no side effect, I take it in combinati...
1    "My son is halfway through his fourth week of ...
2    "I used to take another oral contraceptive, wh...
3    "This is my first time using any form of birth...
4    "Suboxone has completely turned my life around...
Name: review, dtype: object
-------------------------------------------------------------
0    "I&#039;ve tried a few antidepressants over th...
1    "My son has Crohn&#039;s disease and has done ...
2                        "Quick reduction of symptoms"
3    "Contrave combines drugs that were used for al...
4    "I have been on this birth control for one cyc...
Name: review, dtype: object


In [7]:
# Conversion of HTML elements
train_data['review'] = train_data['review'].apply(html.unescape)
test_data['review'] = test_data['review'].apply(html.unescape)

In [8]:
# Lowercase the data
train_data['review'] = train_data['review'].str.lower()
test_data['review'] = test_data['review'].str.lower()

In [9]:
# Tokenization of the reviews
train_data['tokens'] = train_data['review'].apply(word_tokenize)
test_data['tokens'] = test_data['review'].apply(word_tokenize)

In [10]:
# Remove Non-Alphanumerics
import string
train_data['tokens'] = train_data['tokens'].apply(
    lambda x: [word for word in x if word.isalnum()]
)
test_data['tokens'] = test_data['tokens'].apply(
    lambda x: [word for word in x if word.isalnum()]
)

In [11]:
# Remove Stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
train_data['tokens'] = train_data['tokens'].apply(
    lambda x: [word for word in x if word not in stop_words]
)
test_data['tokens'] = test_data['tokens'].apply(
    lambda x: [word for word in x if word not in stop_words]
)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
train_data.head()

,Unnamed: 0,drugName,condition,review,rating,date,usefulCount,tokens
0,206461,Valsartan,Left Ventricular Dysfunction,"""it has no side effect, i take it in combinati...",9.0,"May 20, 2012",27,"[side, effect, take, combination, bystolic, 5,..."
1,95260,Guanfacine,ADHD,"""my son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,"[son, halfway, fourth, week, intuniv, became, ..."
2,92703,Lybrel,Birth Control,"""i used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,"[used, take, another, oral, contraceptive, 21,..."
3,138000,Ortho Evra,Birth Control,"""this is my first time using any form of birth...",8.0,"November 3, 2015",10,"[first, time, using, form, birth, control, gla..."
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""suboxone has completely turned my life around...",9.0,"November 27, 2016",37,"[suboxone, completely, turned, life, around, f..."


In [13]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

!unzip -q -o /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:
from nltk.stem import WordNetLemmatizer
# Lemmatization
lemmatizer = WordNetLemmatizer()
train_data['tokens'] = train_data['tokens'].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x]
)
test_data['tokens'] = test_data['tokens'].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x]
)

In [15]:
# Saving preprocessed data
train_data.to_csv('/kaggle/working/preprocessed_train.csv', index=False)
test_data.to_csv('/kaggle/working/preprocessed_test.csv', index=False)